# Notes:

Metrics to add:
- column for type of workout so we can predict in different workouts


Time sequence options:
- pad at the end of each sequence and build an LSTM RNN with a layer that ignores padded portions of the sequence and a bidirectional layer to retain context from each phase
    1) separate pitch into phases (wind up, sit down power, and throw maybe? or possibly: wind-up, stride, arm cocking, acceleration, release)
        - ENSURE phases are always started the same, this takes away the noise of different starts and is ESSENTIAL or just do tree based instead of lstm
        - Add phase ID (e.g., wind_up=0, acceleration=1, release=2) as a feature to the model.
    2) use time column to split into different sequences (per trial)
        - consider normalizing the time sequences to 0,1. This only makes sense if it's not already normalized, check first and check distance in time steps to ensure no noise
    3) pad at the end of each phase so that it each sequence matches the max sequence
        - Pad each phase separately to its own max length (e.g., wind-up padded to 20 steps, acceleration to 50 steps).
        - Use a hierarchical model (e.g., separate LSTM per phase) to avoid mixing padded values across phases.
    4) create masked neural network that is bidirectional for edge cases and masking so we exclude the padded steps
        For causal tasks, use unidirectional LSTMs.
        For non-causal tasks (e.g., post-pitch classification), use bidirectional layers.
            from tensorflow.keras.layers import Input, Masking, Bidirectional, LSTM, Dense

            # Phase-specific input (e.g., wind-up)
            inputs = Input(shape=(max_phase_length, num_features + 2))  # +2 for phase ID and normalized time
            masked = Masking(mask_value=-1.0)(inputs)  # Assume padded time = -1
            lstm_out = Bidirectional(LSTM(64))(masked)  
            outputs = Dense(1, activation='sigmoid')(lstm_out)
        Add attention mechanisms to explicitly highlight critical moments (e.g., release):
            python
            from tensorflow.keras.layers import Attention, Concatenate
            # Add attention to focus on key timesteps (e.g., release)
            context_vector = Attention()([lstm_out, lstm_out])
            outputs = Concatenate()([lstm_out, context_vector])


Other options to test
- Dynamic Time Warping (DTW): Align sequences non-linearly to a reference length.
    Use Case: Best for comparing shapes of biomechanical curves (e.g., elbow angle vs. time).
    Use DTW as a preprocessing step for similarity-based tasks (e.g., clustering pitches by motion pattern).

- Resampling: Interpolate sequences to a fixed length (e.g., 100 steps) using biomechanical curves’ shape.
    Use Case: Suitable when biomechanical patterns are time-invariant (e.g., joint angles during release).
    Use cubic spline interpolation instead of linear to preserve curve shape.


Key Tests to Validate Approach

    Phase Alignment Check:

        Plot the distribution of phase durations (e.g., wind-up: 0.2–0.3s, release: 0.1–0.15s).

        Ensure padding doesn’t exceed 20% of the phase’s natural duration.
Key Tests to Validate Approach

    Phase Alignment Check:

        Plot the distribution of phase durations (e.g., wind-up: 0.2–0.3s, release: 0.1–0.15s).

        Ensure padding doesn’t exceed 20% of the phase’s natural duration.

    Ablation Study:

        Compare performance of:
        a) Phase-padded LSTM
        b) Resampled LSTM
        c) DTW-aligned model

    Attention Visualization:

        Use libraries like tf-explain to confirm attention focuses on biomechanically critical moments (e.g., release).

    Noise Injection Test:

        Add Gaussian noise to padded regions and verify model robustness (accuracy shouldn’t degrade)
    Ablation Study:

        Compare performance of:
        a) Phase-padded LSTM
        b) Resampled LSTM
        c) DTW-aligned model

    Attention Visualization:

        Use libraries like tf-explain to confirm attention focuses on biomechanically critical moments (e.g., release).

    Noise Injection Test:

        Add Gaussian noise to padded regions and verify model robustness (accuracy shouldn’t degrade)

# Trials analysis

In [15]:
import pandas as pd
import random
import re
from datetime import datetime, timedelta

# ============================================================================
# Step 1: Extract Filename Timestamps in the Trials Dataset
# ----------------------------------------------------------------------------
def extract_filename_timestamps(trials_df):
    """
    If a 'filename' column exists in trials_df, extract the datetime stamp 
    from each filename using two regex patterns. This adds several new columns:
      - extracted_datetime: the parsed datetime object from the filename.
      - extracted_raw: the raw timestamp string.
      - extraction_pattern: which pattern matched.
      - time_parsed: conversion of the 'time' column to a datetime.
      - time_difference: difference between time_parsed and extracted_datetime.
      - extracted_date: the date part of extracted_datetime.
      - extracted_time: the time part of extracted_datetime.
    """
    if 'filename' in trials_df.columns:
        print("\n==== Filename Timestamp Extraction and Comparison ====")
        
        # Define regex patterns.
        pattern1 = re.compile(r'^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})')
        pattern2 = re.compile(r'^(\d{8}_\d{6})')
        
        def extract_filename_timestamp(fname):
            """
            Try to extract a datetime stamp from a filename using two patterns.
            Returns a tuple: (datetime_object or None, raw_timestamp_string or None, pattern_type)
            """
            match1 = pattern1.match(fname)
            if match1:
                raw_ts = match1.group(1)
                try:
                    dt_obj = datetime.strptime(raw_ts, "%Y_%m_%d_%H_%M_%S")
                    return dt_obj, raw_ts, "underscore-type"
                except Exception as e:
                    return None, raw_ts, "underscore-type"
            match2 = pattern2.match(fname)
            if match2:
                raw_ts = match2.group(1)
                try:
                    dt_obj = datetime.strptime(raw_ts, "%Y%m%d_%H%M%S")
                    return dt_obj, raw_ts, "compact-type"
                except Exception as e:
                    return None, raw_ts, "compact-type"
            return None, None, "no-match"
        
        # Apply extraction to each filename.
        trials_df['extracted_datetime'] = trials_df['filename'].apply(lambda x: extract_filename_timestamp(str(x))[0])
        trials_df['extracted_raw'] = trials_df['filename'].apply(lambda x: extract_filename_timestamp(str(x))[1])
        trials_df['extraction_pattern'] = trials_df['filename'].apply(lambda x: extract_filename_timestamp(str(x))[2])
        
        print("Filename pattern counts:")
        print(trials_df['extraction_pattern'].value_counts())
        
        print("\nSample records for underscore-type:")
        print(trials_df[trials_df['extraction_pattern'] == 'underscore-type'][['filename', 'extracted_raw', 'extracted_datetime']].head())
        
        print("\nSample records for compact-type:")
        print(trials_df[trials_df['extraction_pattern'] == 'compact-type'][['filename', 'extracted_raw', 'extracted_datetime']].head())
        
        # Compare extracted_datetime with the trials 'time' column.
        trials_df['time_parsed'] = pd.to_datetime(trials_df['time'], errors='coerce')
        trials_df['time_difference'] = trials_df['time_parsed'] - trials_df['extracted_datetime']
        
        print("\nSample comparison between 'time_parsed' and 'extracted_datetime':")
        print(trials_df[['time', 'time_parsed', 'extracted_datetime', 'time_difference']].head())
        
        # Extract date and time parts from extracted_datetime.
        trials_df['extracted_date'] = trials_df['extracted_datetime'].dt.date
        trials_df['extracted_time'] = trials_df['extracted_datetime'].dt.time
        
        print("\nSample extracted date and time from filename:")
        print(trials_df[['extracted_datetime', 'extracted_date', 'extracted_time']].head())
    else:
        print("\nNo 'filename' column found in trials dataset.")
    return trials_df

# ============================================================================
# Step 2: Existing Trials Analysis and Deduplication Functions
# ----------------------------------------------------------------------------
def analyze_trials_duplicates(trials_df, trial_id='498_2'):
    """
    Comprehensive analysis of the trials dataset for duplicates or data loss.
    Also prints detailed information for one specific session_trial.
    Compares summary statistics before and after deduplication.
    """
    print("\n=== Complete Trials Dataset Analysis ===")
    
    # Display column data types.
    print("\nColumn Data Types:")
    for col in trials_df.columns:
        print(f"{col}: {trials_df[col].dtype}")
    
    # Duplicate Analysis.
    print("\n=== Duplicate Analysis ===")
    print(f"Total rows: {len(trials_df)}")
    print(f"Unique session_trials: {trials_df['session_trial'].nunique()}")
    
    # Null Value Analysis.
    print("\n=== Null Value Analysis ===")
    null_counts = trials_df.isnull().sum()
    print("\nColumns with null values:")
    print(null_counts[null_counts > 0])
    
    # Categorical Column Value Counts.
    print("\n=== Categorical Column Value Counts ===")
    categorical_cols = trials_df.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        unique_count = trials_df[col].nunique()
        if unique_count < 50:
            print(f"\n{col} - Unique values: {unique_count}")
            print(trials_df[col].value_counts().head())
    
    # Check for multiple values per session_trial.
    print("\n=== Multiple Values per Session Trial Check ===")
    for col in trials_df.columns:
        values_per_trial = trials_df.groupby('session_trial')[col].nunique()
        if values_per_trial.max() > 1:
            print(f"\nColumn '{col}' has multiple values for some session_trials:")
            print(f"Max values per trial: {values_per_trial.max()}")
            print("Example trials with multiple values:")
            multi_value_trials = values_per_trial[values_per_trial > 1].index[:3]
            for trial in multi_value_trials:
                print(f"\nTrial {trial}:")
                print(trials_df[trials_df['session_trial'] == trial][col].unique())
    
    # Summary Statistics Before Deduplication.
    print("\n=== Summary Statistics Before Dropping Duplicates ===")
    numeric_cols = trials_df.select_dtypes(include=['number']).columns
    if not numeric_cols.empty:
        print(trials_df[numeric_cols].describe())
    else:
        print("No numeric columns found.")
    
    # Impact of Dropping Duplicates.
    print("\n=== Impact of Dropping Duplicates (All Columns) ===")
    before_drop = trials_df.copy()
    after_drop = trials_df.drop_duplicates(subset=['session_trial'])
    
    print(f"\nBefore dropping duplicates: Total rows: {len(before_drop)}; Memory usage: {before_drop.memory_usage().sum() / 1024**2:.2f} MB")
    print(f"After dropping duplicates: Total rows: {len(after_drop)}; Memory usage: {after_drop.memory_usage().sum() / 1024**2:.2f} MB")
    
    # Summary Statistics After Deduplication.
    print("\n=== Summary Statistics After Dropping Duplicates ===")
    if not numeric_cols.empty:
        print(after_drop[numeric_cols].describe())
    else:
        print("No numeric columns found.")
    
    # Value Distribution Changes.
    print("\n=== Value Distribution Changes ===")
    for col in trials_df.columns:
        before_unique = set(before_drop[col].dropna().unique())
        after_unique = set(after_drop[col].dropna().unique())
        if before_unique != after_unique:
            print(f"\nWarning: Column '{col}' has different values after dropping duplicates")
            print("Values only in original:", before_unique - after_unique)
            print("Values only in deduplicated:", after_unique - before_unique)
    
    # Detailed Check on One Specific session_trial.
    print("\n=== Detailed Check on session_trial '{}' ===".format(trial_id))
    original_trial = trials_df[trials_df['session_trial'] == trial_id]
    dedup_trial = after_drop[after_drop['session_trial'] == trial_id]
    
    print("\nOriginal rows for session_trial '{}':".format(trial_id))
    print(original_trial)
    print("\nUnique values per column in original data for '{}':".format(trial_id))
    for col in trials_df.columns:
        unique_vals = original_trial[col].dropna().unique()
        print(f"{col}: {unique_vals}")
    
    print("\nDeduplicated row for session_trial '{}':".format(trial_id))
    print(dedup_trial)
    
    return after_drop

# ============================================================================
# Step 3: Additional Validation Functions for Merge and Time Checks
# ----------------------------------------------------------------------------
def validate_merge(trials_unique, joint_moments, merged_df):
    """
    Validates that merging joint_moments with deduplicated trials metadata
    did not change the trial-level distribution.
    Also performs a random sampling check on a few session_trial values.
    """
    print("\n=== Merge Validation ===")
    print("Number of rows in joint_moments:", len(joint_moments))
    print("Number of rows in merged_df:", len(merged_df))
    missing_trial_info = merged_df['trials_id'].isnull().sum()
    print("Rows in joint_moments with missing trial info after merge:", missing_trial_info)
    
    print("\n=== Summary Statistics for Trial Metadata in Merged Data ===")
    numeric_cols = trials_unique.select_dtypes(include=['number']).columns
    if not numeric_cols.empty:
        merged_stats = merged_df[numeric_cols].describe()
        original_stats = trials_unique[numeric_cols].describe()
        print("Merged Trial Metadata Statistics:")
        print(merged_stats)
        print("\nOriginal Deduplicated Trials Statistics:")
        print(original_stats)
    else:
        print("No numeric trial metadata columns to compare.")
    
    print("\n=== Random Sampling Check on session_trial values ===")
    sample_ids = random.sample(list(joint_moments['session_trial'].unique()), 
                               min(5, joint_moments['session_trial'].nunique()))
    for sid in sample_ids:
        original_metadata = trials_unique[trials_unique['session_trial'] == sid]
        merged_metadata = merged_df[merged_df['session_trial'] == sid]
        print(f"\nSession_trial: {sid}")
        print("Original Metadata:")
        print(original_metadata)
        if not merged_metadata.empty:
            merged_row = merged_metadata.iloc[0][trials_unique.columns]
            print("Merged Metadata (first row):")
            print(merged_row)
        else:
            print("No merged rows found for this session_trial!")

def validate_time_columns(trials, joint_moments):
    """
    Validates the conversion of the 'time' columns for both trials and joint_moments.
    """
    print("\n=== Time Column Conversion Check ===")
    try:
        trials_time_parsed = pd.to_timedelta(trials['time'], errors='coerce')
        print("Trials 'time' conversion: min =", trials_time_parsed.min(), "max =", trials_time_parsed.max())
    except Exception as e:
        print("Error converting trials 'time':", e)
    
    try:
        joint_time_parsed = pd.to_timedelta(joint_moments['time'], errors='coerce')
        print("Joint_moments 'time' conversion: min =", joint_time_parsed.min(), "max =", joint_time_parsed.max())
    except Exception as e:
        print("Error converting joint_moments 'time':", e)

def validate_cross_table_consistency(trials, joint_moments):
    """
    Validates that all session_trial values in joint_moments are present in trials.
    """
    trials_set = set(trials['session_trial'].unique())
    joint_set = set(joint_moments['session_trial'].unique())
    missing_in_trials = joint_set - trials_set
    if missing_in_trials:
        print("\nWARNING: The following session_trial values are in joint_moments but not in trials:")
        print(missing_in_trials)
    else:
        print("\nAll session_trial values in joint_moments are present in trials.")

# ============================================================================
# Step 4: Demonstrate Difference Between Inner Join and Left Join
# ----------------------------------------------------------------------------
def demonstrate_join_types(joint_moments, trials_unique):
    """
    Demonstrates the difference between inner joining and left joining joint_moments
    with the trials metadata.
    """
    merge_left = pd.merge(joint_moments, trials_unique, on='session_trial', how='left', suffixes=('_joint', '_trial'))
    merge_inner = pd.merge(joint_moments, trials_unique, on='session_trial', how='inner', suffixes=('_joint', '_trial'))
    
    print("\n=== Join Types Comparison ===")
    print("Rows in joint_moments:", len(joint_moments))
    print("Rows in left join merge:", len(merge_left))
    print("Rows in inner join merge:", len(merge_inner))
    print("\nNote: An inner join will drop joint_moments rows without matching session_trial in trials.")
    return merge_left, merge_inner

# ============================================================================
# Step 5: Create Time Series Features for the ML Pipeline
# ----------------------------------------------------------------------------
def prepare_time_series_features(merged_df):
    """
    For each record in merged_df, create new time series features:
      - 'time_offset': numeric offset (in seconds) from the start.
      - 'ongoing_datetime': extracted_datetime + time_offset.
      - 'duration_from_start': time_offset as a timedelta.
      - Also split extracted_datetime into components: month, day, hour, minute, second, weekday, week.
    """
    # Convert 'time' to numeric seconds.
    merged_df['time_offset'] = pd.to_numeric(merged_df['time'], errors='coerce')
    
    # Create 'ongoing_datetime' = extracted_datetime + time offset.
    merged_df['ongoing_datetime'] = merged_df.apply(
        lambda row: row['extracted_datetime'] + timedelta(seconds=row['time_offset'])
        if pd.notnull(row['extracted_datetime']) and pd.notnull(row['time_offset'])
        else pd.NaT, axis=1)
    
    # Create 'duration_from_start' as a timedelta.
    merged_df['duration_from_start'] = merged_df['time_offset'].apply(lambda x: timedelta(seconds=x) if pd.notnull(x) else pd.NaT)
    
    # Split extracted_datetime into its components (if not null).
    merged_df['extracted_month'] = merged_df['extracted_datetime'].dt.month
    merged_df['extracted_day'] = merged_df['extracted_datetime'].dt.day
    merged_df['extracted_hour'] = merged_df['extracted_datetime'].dt.hour
    merged_df['extracted_minute'] = merged_df['extracted_datetime'].dt.minute
    merged_df['extracted_second'] = merged_df['extracted_datetime'].dt.second
    merged_df['extracted_weekday'] = merged_df['extracted_datetime'].dt.dayofweek  # Monday=0, Sunday=6
    # Using isocalendar() to get ISO week number (as integer)
    merged_df['extracted_week'] = merged_df['extracted_datetime'].dt.isocalendar().week
    
    print("\n=== Ongoing Datetime Feature Summary ===")
    print("ongoing_datetime stats:")
    print(merged_df['ongoing_datetime'].describe())
    
    print("\nDuration from start (timedelta) stats:")
    print(merged_df['duration_from_start'].describe())
    
    print("\nSample of ongoing_datetime and duration_from_start values:")
    print(merged_df[['session_trial', 'extracted_datetime', 'time_offset', 'ongoing_datetime', 'duration_from_start']].head())
    
    print("\nExtracted DateTime Components (from extracted_datetime):")
    print(merged_df[['session_trial', 'extracted_datetime', 'extracted_month', 'extracted_day', 'extracted_hour', 'extracted_minute', 'extracted_second', 'extracted_weekday', 'extracted_week']].head())
    
    return merged_df

# ============================================================================
# Step 6: New Function to Explore the Merged ML Dataset
# ----------------------------------------------------------------------------
def explore_ml_dataset(merged_df):
    """
    Explores the left joined merged dataset to provide a comprehensive overview:
      - Total unique throws.
      - Total records and average per throw.
      - Distributions of extracted_datetime and ongoing_datetime.
      - Counts of key categorical fields.
      - Summary statistics for joint measurement columns.
      - Splitted datetime component features.
      - Final null value counts for each column.
      - A sample of the dataset.
    """
    print("\n=== Exploring the ML Dataset ===")
    
    # Unique throws and record counts.
    unique_throws = merged_df['session_trial'].nunique()
    total_rows = len(merged_df)
    print("Total unique throws (session_trial):", unique_throws)
    print("Total number of records (joint measurements):", total_rows)
    print("Average records per throw:", total_rows / unique_throws if unique_throws > 0 else 0)
    
    # Distribution of extracted_datetime.
    print("\n=== Distribution of Extracted Datetime ===")
    if merged_df['extracted_datetime'].notnull().any():
        print(merged_df['extracted_datetime'].describe())
    else:
        print("No extracted_datetime values present.")
    
    # Distribution of ongoing_datetime.
    print("\n=== Distribution of Ongoing Datetime ===")
    if merged_df['ongoing_datetime'].notnull().any():
        print(merged_df['ongoing_datetime'].describe())
    else:
        print("No ongoing_datetime values present.")
    
    # Categorical fields counts.
    print("\n=== Categorical Fields Counts (from Trials Metadata) ===")
    for col in ['session_type', 'pitch_type', 'handedness']:
        if col in merged_df.columns:
            print(f"\n{col} value counts:")
            print(merged_df[col].value_counts())
    
    # Summary statistics for joint measurement columns.
    moment_columns = [col for col in merged_df.columns if "moment" in col.lower()]
    if moment_columns:
        print("\n=== Summary Statistics for Joint Moment Columns ===")
        print(merged_df[moment_columns].describe())
    else:
        print("\nNo joint moment columns detected in merged dataset.")
    
    # Show a sample of the dataset.
    print("\n=== Sample Rows from the Merged Dataset ===")
    print(merged_df.head())
    
    # Final null value counts.
    print("\n=== Final Null Value Counts in Merged Dataset ===")
    print(merged_df.isnull().sum())
    
    # Context reminder.
    print("\n=== Context and Available Tables ===")
    print("Based on the uploaded schema files (biomech_pitching_db_v7.txt and theia_pitching_db.txt),\n"
          "there are multiple joint measurement tables (e.g. joint_moments, joint_forces, joint_velos) and \n"
          "trial-level metadata in the 'trials' table. In this merged dataset, joint_moments is combined with \n"
          "trial metadata via session_trial. We have added derived time series features ('ongoing_datetime' and \n"
          "'duration_from_start') as well as split components of the start timestamp. These features can be used \n"
          "as the time axis for a forecasting pipeline or as input features for tree-based algorithms.\n"
          "Additionally, you have both an absolute time (ongoing_datetime) and a relative duration (duration_from_start)\n"
          "that can help capture different aspects of the pitching motion dynamics.")
    
    return merged_df

# Load datasets

import pandas as pd
import os

# Set the data directory path
data_dir = '../../data/processed/sql_datasets/'
output_merged_dataset = '../../data/processed/sql_datasets/merged_examples/joint_trials_sessions_merged_datasets.csv'
# Get list of all CSV files in the directory
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

# Create a dictionary to store all dataframes
datasets = {}

# Load each CSV file into a dataframe
for file in csv_files:
    # Get name without extension to use as dict key
    name = os.path.splitext(file)[0]
    # Read CSV and store in dictionary
    datasets[name] = pd.read_csv(os.path.join(data_dir, file))
    print(f"{name} loaded with shape {datasets[name].shape}")

# Print the names of all loaded datasets
print("\nDatasets loaded:")
print(", ".join(datasets.keys()))

# ============================================================================
# Step 7: Main Execution Block with Filtering on '498_' Session Trials
# ----------------------------------------------------------------------------

# Retrieve the trials and joint_moments datasets from the loaded dictionary.
trials_sample = datasets.get('trials_sample')
joint_moments_sample = datasets.get('joint_moments_sample')

if trials_sample is None or joint_moments_sample is None:
    print("One or more required datasets ('trials' and 'joint_moments') are missing.")
else:
    # --- Step A: Add extracted_datetime to trials.
    trials = extract_filename_timestamps(trials_sample)
    
    # --- Step B: Analyze and deduplicate trials.
    deduplicated_trials = analyze_trials_duplicates(trials)
    
    # --- Step C: Filter to session_trial values starting with '498_'.
    trials_498 = deduplicated_trials[deduplicated_trials['session_trial'].str.startswith("498_")]
    joint_moments_498 = joint_moments_sample[joint_moments_sample['session_trial'].str.startswith("498_")]
    
    print("\n=== Filtering to session_trial values starting with '498_' ===")
    print("Number of trials (filtered):", len(trials_498))
    print("Number of joint_moments (filtered):", len(joint_moments_498))
    
    # --- Step D: Validate cross-table consistency BEFORE merging.
    validate_cross_table_consistency(trials_498, joint_moments_498)
    
    # --- Step E: Validate time column conversion.
    validate_time_columns(trials_498, joint_moments_498)
    
    # --- Step F: Create deduplicated trials metadata for merge.
    trials_unique_498 = trials_498[['session_trial', 'trials_id', 'session', 'trial', 
                                    'pitch_type', 'session_type', 'handedness', 'extracted_datetime']]
    
    # --- Step G: Merge joint_moments_498 with trials_unique_498 using a LEFT join.
    merged_df_left = pd.merge(joint_moments_498, trials_unique_498, on='session_trial', how='left', 
                              suffixes=('_joint', '_trial'))
    print("\n=== Merge Validation (LEFT JOIN on '498_' filtered data) ===")
    print("Number of rows in joint_moments_498:", len(joint_moments_498))
    print("Number of rows in merged_df_left:", len(merged_df_left))
    missing_trial_info = merged_df_left['trials_id'].isnull().sum()
    print("Rows with missing trial info after left join:", missing_trial_info)
    
    # --- Step H: Demonstrate inner join vs left join on the filtered data.
    merged_df_left, merged_df_inner = demonstrate_join_types(joint_moments_498, trials_unique_498)
    
    # --- Step I: Validate the merge.
    validate_merge(trials_unique_498, joint_moments_498, merged_df_left)
    
    # --- Step J: Create additional time series features (ongoing_datetime, duration_from_start, and split datetime components)
    merged_df_left = prepare_time_series_features(merged_df_left)
    
    # --- Step K: Explore the ML dataset.
    merged_df_left = explore_ml_dataset(merged_df_left)
    
    # --- Step L: Save the merged dataset.
    merged_df_left.to_csv(output_merged_dataset, index=False)


joint_forces_sample loaded with shape (10000, 27)
joint_moments_sample loaded with shape (10000, 27)
joint_velos_sample loaded with shape (10000, 42)
sessions_sample loaded with shape (2714, 19)
trials_sample loaded with shape (10000, 26)

Datasets loaded:
joint_forces_sample, joint_moments_sample, joint_velos_sample, sessions_sample, trials_sample

==== Filename Timestamp Extraction and Comparison ====
Filename pattern counts:
extraction_pattern
underscore-type    6083
compact-type       3917
Name: count, dtype: int64

Sample records for underscore-type:
                                               filename        extracted_raw  \
3917  2024_02_01_12_52_34_joemarsh_000167_PitchingAs...  2024_02_01_12_52_34   
3918  2024_02_01_12_52_42_joemarsh_000167_PitchingAs...  2024_02_01_12_52_42   
3919  2024_02_01_12_52_50_joemarsh_000167_PitchingAs...  2024_02_01_12_52_50   
3920  2024_02_01_12_52_58_joemarsh_000167_PitchingAs...  2024_02_01_12_52_58   
3921  2024_02_01_12_58_55_joemarsh_000

C:\Users\ghadf\AppData\Local\Temp\ipykernel_5568\3477164314.py:66: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  trials_df['time_parsed'] = pd.to_datetime(trials_df['time'], errors='coerce')



Sample comparison between 'time_parsed' and 'extracted_datetime':
  time time_parsed  extracted_datetime time_difference
0  NaN         NaT 2023-01-31 09:57:14             NaT
1  NaN         NaT 2023-01-31 09:57:14             NaT
2  NaN         NaT 2023-01-31 09:57:14             NaT
3  NaN         NaT 2023-01-31 09:57:14             NaT
4  NaN         NaT 2023-01-31 09:57:14             NaT

Sample extracted date and time from filename:
   extracted_datetime extracted_date extracted_time
0 2023-01-31 09:57:14     2023-01-31       09:57:14
1 2023-01-31 09:57:14     2023-01-31       09:57:14
2 2023-01-31 09:57:14     2023-01-31       09:57:14
3 2023-01-31 09:57:14     2023-01-31       09:57:14
4 2023-01-31 09:57:14     2023-01-31       09:57:14

=== Complete Trials Dataset Analysis ===

Column Data Types:
trials_id: int64
session: int64
trial: int64
session_trial: object
time: object
session_type: object
tag: object
pitch_type: object
ball_weight: float64
drill_tag: float64
style_tag:

# Merge with EMG metrics
- Load From:
    * emg_data_parse_and_load_final.ipynb validates and parses the emg data with the timestamp added on, will add on bulk loading and kafka datastreaming so it's better for this case
Things to watch out for:
Tolerance in join difference? should it be within .0001 sec or how big would we set up tolerance to match up those different systems?
Validation to ensure we aren't losing any metrics from either


possibly this:
Alternative Approach: Using merge_asof for Nearest-Time Joins

If the datasets have timestamps that are close but not identical and you’d prefer to merge based on the nearest time (within a certain tolerance), you can use pd.merge_asof.

# Sort dataframes by datetime
df1 = df1.sort_values('datetime')
df2 = df2.sort_values('datetime')

# Perform an asof merge: This will match each row in df1 to the nearest row in df2
merged_asof = pd.merge_asof(df1, df2, on='datetime', direction='nearest', tolerance=pd.Timedelta('1s'))

Note: With merge_asof, it’s important that the data is sorted by the datetime column, and you can set a tolerance (e.g., one second) to control how far apart the matching timestamps can be.

In [ ]:
import pandas as pd
import numpy as np

# -------------------------------
# Step 1: Inspect and Prepare DataFrames
# -------------------------------

# Debug: Print columns and a sample of merged_df_left (base dataset)
print("merged_df_left columns:")
print(merged_df_left.columns)
print("\nmerged_df_left sample data:")
print(merged_df_left.head())

# Debug: Print columns and a sample of EMG_metrics (original)
print("\nEMG_metrics columns:")
print(EMG_metrics.columns)
print("\nEMG_metrics sample data:")
print(EMG_metrics.head())

# Preliminary: Sort EMG_metrics by 'Timestamp' and compute time step differences
EMG_metrics = EMG_metrics.sort_values('Timestamp')
EMG_metrics['timestamp_step'] = pd.to_datetime(EMG_metrics['Timestamp']).diff()
print("\nTime step differences in EMG_metrics based on Timestamp (first 10 rows):")
print(EMG_metrics[['Timestamp', 'timestamp_step']].head(10))

# Debug: Check for duplicate timestamps
duplicate_timestamps = EMG_metrics[EMG_metrics.duplicated('Timestamp', keep=False)]
print("\nNumber of duplicate timestamps:", len(duplicate_timestamps))

# -------------------------------
# Step 2: Prepare Datetime Columns for Base Dataset
# -------------------------------

if 'ongoing_datetime' in merged_df_left.columns:
    merged_df_left['ongoing_datetime'] = pd.to_datetime(merged_df_left['ongoing_datetime'])
    merged_df_left['datetime'] = merged_df_left['ongoing_datetime']
else:
    raise ValueError("merged_df_left does not contain a suitable datetime column (e.g., 'ongoing_datetime').")

print("\nBase dataset datetime sample:")
print(merged_df_left[['ongoing_datetime', 'datetime']].head())

# -------------------------------
# Step 3: Prepare Datetime Columns in EMG_metrics
# -------------------------------

if 'Date/Time' in EMG_metrics.columns and 'Timestamp' in EMG_metrics.columns:
    EMG_metrics['Date/Time_parsed'] = pd.to_datetime(EMG_metrics['Date/Time'])
    EMG_metrics['Timestamp_parsed'] = pd.to_datetime(EMG_metrics['Timestamp'])
    
    print("\nComparison between 'Date/Time' and 'Timestamp':")
    print(EMG_metrics[['Date/Time', 'Timestamp', 'Date/Time_parsed', 'Timestamp_parsed']].head())
    
    # Use the dynamic Timestamp for merging:
    EMG_metrics['emg_time'] = EMG_metrics['Timestamp']
    EMG_metrics['datetime'] = EMG_metrics['Timestamp_parsed']
    
elif 'Date/Time' in EMG_metrics.columns:
    EMG_metrics['emg_time'] = EMG_metrics['Date/Time']
    EMG_metrics['datetime'] = pd.to_datetime(EMG_metrics['Date/Time'])
elif 'Timestamp' in EMG_metrics.columns:
    EMG_metrics['emg_time'] = EMG_metrics['Timestamp']
    EMG_metrics['datetime'] = pd.to_datetime(EMG_metrics['Timestamp'])
else:
    raise ValueError("EMG_metrics does not contain a suitable datetime column (e.g., 'Date/Time' or 'Timestamp').")

print("\nmerged_df_left datetime range:",
      merged_df_left['datetime'].min(), "to", merged_df_left['datetime'].max())
print("EMG_metrics datetime range:",
      EMG_metrics['datetime'].min(), "to", EMG_metrics['datetime'].max())
print("EMG_metrics Timestamp range:",
      EMG_metrics['Timestamp'].min(), "to", EMG_metrics['Timestamp'].max())

# -------------------------------
# Step 4: Compute Time Steps for merged_df_left
# -------------------------------

merged_df_left['time_step'] = merged_df_left['datetime'].diff()
print("\nTime step differences in merged_df_left (first 10 rows):")
print(merged_df_left[['datetime', 'time_step']].head(10))

# -------------------------------
# Step 5: Sort Both DataFrames by Their Datetime Columns
# -------------------------------

merged_df_left = merged_df_left.sort_values('datetime')
EMG_metrics = EMG_metrics.sort_values('datetime')

print("\nRow count of merged_df_left:", merged_df_left.shape[0])
print("Row count of original EMG_metrics:", EMG_metrics.shape[0])

# -------------------------------
# Step 6: Simulate EMG Data to Span the Same Time Frame as merged_df_left
# -------------------------------

start_time = merged_df_left['datetime'].min()
end_time = merged_df_left['datetime'].max()
print("\nSimulated EMG data will span from:", start_time, "to", end_time)

# Use a frequency based on observed EMG time step (~5ms)
simulated_freq = '5ms'
simulated_time_index = pd.date_range(start=start_time, end=end_time, freq=simulated_freq)
print("\nSimulated EMG data time index length:", len(simulated_time_index))

if 'EMG 1 (mV) - FDS (81770)' in EMG_metrics.columns:
    emg_mean = EMG_metrics['EMG 1 (mV) - FDS (81770)'].mean()
    emg_std = EMG_metrics['EMG 1 (mV) - FDS (81770)'].std()
else:
    emg_mean = 0
    emg_std = 1

simulated_EMG = pd.DataFrame({
    'emg_simulated': np.random.normal(emg_mean, emg_std, len(simulated_time_index))
}, index=simulated_time_index).reset_index().rename(columns={'index': 'datetime'})

simulated_EMG['emg_time'] = simulated_EMG['datetime']

print("\nSimulated EMG data shape:", simulated_EMG.shape)
print("First 5 rows of simulated EMG data:")
print(simulated_EMG.head())

# -------------------------------
# Step 7: Merge Approaches
# -------------------------------

# 7.1) Merge_asof join using a 10ms tolerance.
merged_simulated_asof = pd.merge_asof(
    merged_df_left,
    simulated_EMG,
    on='datetime',
    direction='nearest',
    tolerance=pd.Timedelta('10ms')
)

# Debug: Check merge_asof join results and row count
print("\nAfter merge_asof join:")
print("Row count:", merged_simulated_asof.shape[0])
print(merged_simulated_asof.head())

# Compute time difference for each merged row
merged_simulated_asof['time_diff'] = (merged_simulated_asof['datetime'] - 
                                        pd.to_datetime(merged_simulated_asof['emg_time'])).abs()
print("\nTime differences in merge_asof join (first 10 rows):")
print(merged_simulated_asof[['datetime', 'emg_time', 'time_diff']].head(10))

# 7.2) Filter: Keep only rows where time_diff is within 10ms (1-to-1 relationship)
max_tol = pd.Timedelta('10ms')
merged_simulated_filtered = merged_simulated_asof[merged_simulated_asof['time_diff'] <= max_tol]

print("\nAfter filtering for time_diff <= 10ms:")
print("Row count:", merged_simulated_filtered.shape[0])
print(merged_simulated_filtered.head())

# -------------------------------
# (Optional) Outer join approach is skipped here since our filtering creates a 1-to-1 match.
# could outer join to not lose any data but would need to interpolate or input estimate datapoints in between missing sections in time. Not a bad way to go but would have to be specific AF
# -------------------------------


# -------------------------------
# For adding Data onto the muscular data if we wanted to put the phases on there for pure muscular data prediction
# -------------------------------

# Suppose muscular_data is your DataFrame containing muscular measurements.
# It should have a datetime column – for example, 'muscular_datetime'.
# We'll simulate a small muscular_data DataFrame for demonstration:

# For demonstration, let's create a simulated muscular_data DataFrame:
muscular_time_index = pd.date_range(start=merged_df_left['datetime'].min(),
                                    end=merged_df_left['datetime'].max(),
                                    freq='10ms')  # suppose muscular data is sampled every 10ms
muscular_data = pd.DataFrame({
    'muscular_value': np.random.normal(100, 5, len(muscular_time_index))
}, index=muscular_time_index).reset_index().rename(columns={'index': 'muscular_datetime'})

# Convert and sort the muscular data datetime column:
muscular_data['muscular_datetime'] = pd.to_datetime(muscular_data['muscular_datetime'])
muscular_data = muscular_data.sort_values('muscular_datetime')

print("\nMuscular data sample:")
print(muscular_data.head())

# Now, perform a merge_asof join between our filtered EMG merge (merged_simulated_filtered)
# and the muscular_data.
# We'll join on our base 'datetime' and muscular_data's 'muscular_datetime'.
# Adjust the tolerance based on your data’s granularity (here we set it to 10ms for example):

final_merged = pd.merge_asof(
    merged_simulated_filtered.sort_values('datetime'),
    muscular_data.sort_values('muscular_datetime'),
    left_on='datetime',
    right_on='muscular_datetime',
    direction='nearest',
    tolerance=pd.Timedelta('10ms')
)

print("\nAfter joining with muscular data:")
print("Final merged row count:", final_merged.shape[0])
print(final_merged.head(10))



merged_df_left columns:
Index(['jointmoment_id', 'session_trial', 'time', 'elbow_moment_x',
       'elbow_moment_y', 'elbow_moment_z', 'shoulder_thorax_moment_x',
       'shoulder_thorax_moment_y', 'shoulder_thorax_moment_z',
       'shoulder_upper_arm_moment_x', 'shoulder_upper_arm_moment_y',
       'shoulder_upper_arm_moment_z', 'wrist_moment_x', 'wrist_moment_y',
       'wrist_moment_z', 'glove_elbow_moment_x', 'glove_elbow_moment_y',
       'glove_elbow_moment_z', 'glove_shoulder_glove_upper_arm_moment_x',
       'glove_shoulder_glove_upper_arm_moment_y',
       'glove_shoulder_glove_upper_arm_moment_z',
       'glove_shoulder_thorax_moment_x', 'glove_shoulder_thorax_moment_y',
       'glove_shoulder_thorax_moment_z', 'glove_wrist_moment_x',
       'glove_wrist_moment_y', 'glove_wrist_moment_z', 'trials_id', 'session',
       'trial', 'pitch_type', 'session_type', 'handedness',
       'extracted_datetime', 'time_offset', 'ongoing_datetime',
       'duration_from_start', 'extracted_

EDA
Check all the basics:
- p value for each metric for suspicious stuff
- outliers
- standardization
- normalization
- dispersion
- disparity
- nulls and which to take out or to impute
- anything else that fixes this dataset 

# Feature Engineering

metrics to add:
* Set up phases based on the pitcher movement so we can set them into different phases
* add the phases to the EMG dataset so we can better predict on that and create use the merge_asof dataset to see if it's worth predicting with for the time series motion of the joints (the merge_asof would need to be a small difference)
* add in pulse data if we don't have arm slot/torque/speed in the database
* add in vulgas range and possibly the ongoing detriment from that
* finish up the sql statement (ensure it's completely validated) for the base table
* separate the dataset into time series classification, regression and tree based classification datasets
* check the feaures importances and such
* use permutation importance and use github(dot)com/eli5-org/eli5

Setting up feature dictionary to speak over each feature to talk about actual important metrics outside of those feature selected
Feature Selection (RFE, SHAP importance, correlation (to take out multi-collinearity metrics or combine them))
